# OpenReels · Wan 2.1 VACE 1.3B — Kaggle GPU smoke test

Этот ноутбук проверяет бесплатную GPU-сессию и пытается создать короткий MP4 через ComfyUI API. Нужны включённые **Internet** и **GPU** в настройках Kaggle Notebook. Модели занимают около 10.5 GB на диске. Запуск не гарантирует, что конкретная GPU/сессия выдержит генерацию.

Используется официальный Comfy-Org repack: VACE 1.3B FP16 (4.01 GB), UMT5 FP8 и Wan VAE. Тест короткий: 17 кадров, 832×480, 16 FPS. Не запускать VACE 14B.


In [ ]:
import os, sys, subprocess, shutil, time, json, requests, textwrap
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU не подключена. В Kaggle: Notebook options → Accelerator → GPU, затем Restart session.')

gpu = torch.cuda.get_device_properties(0)
vram_gib = gpu.total_memory / 1024**3
print(f'GPU: {gpu.name}')
print(f'VRAM: {vram_gib:.1f} GiB')
print(f'PyTorch: {torch.__version__}; CUDA: {torch.version.cuda}')
if vram_gib < 12:
    raise RuntimeError(f'Недостаточно VRAM для разумного теста VACE 1.3B: обнаружено {vram_gib:.1f} GiB. Останавливаюсь до загрузки больших файлов.')
print('GPU-проверка пройдена. Это ещё не означает, что генерация точно поместится в память.')

## 1. Установить ComfyUI
Код не меняет системный PyTorch; если зависимости ComfyUI потребуют другую версию, установка может завершиться ошибкой.

In [ ]:
WORK = Path('/kaggle/working')
COMFY = WORK / 'ComfyUI'
if not COMFY.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git',str(COMFY)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(COMFY/'requirements.txt')], check=True)
print('ComfyUI installed at', COMFY)

## 2. Скачать VACE 1.3B и общие компоненты
Загрузка возобновляется, если файл уже существует. Для теста используется FP8 text encoder, чтобы снизить объём памяти.

In [ ]:
MODEL_DIRS = {    'diffusion_models': COMFY/'models/diffusion_models',    'text_encoders': COMFY/'models/text_encoders',    'vae': COMFY/'models/vae',}for p in MODEL_DIRS.values(): p.mkdir(parents=True, exist_ok=True)# Minimum sizes guard against treating an interrupted download as a valid model.FILES = [ ('diffusion_models','wan2.1_vace_1.3B_fp16.safetensors','https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_vace_1.3B_fp16.safetensors', 3_800_000_000), ('text_encoders','umt5_xxl_fp8_e4m3fn_scaled.safetensors','https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', 5_800_000_000), ('vae','wan_2.1_vae.safetensors','https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors', 200_000_000),]for folder, name, url, min_bytes in FILES:    target = MODEL_DIRS[folder] / name    if target.exists() and target.stat().st_size >= min_bytes:        print('Already downloaded:', name, f'{target.stat().st_size/1e9:.2f} GB')        continue    # Never leave a partial file under the final model name.    part = target.with_suffix(target.suffix + '.part')    if part.exists(): part.unlink()    print('Downloading:', name)    try:        with requests.get(url, stream=True, timeout=60) as resp:            resp.raise_for_status()            expected = int(resp.headers.get('content-length', '0') or 0)            written = 0            with open(part, 'wb') as out:                for chunk in resp.iter_content(chunk_size=8*1024*1024):                    if chunk:                        out.write(chunk); written += len(chunk)        if written < min_bytes:            raise RuntimeError(f'{name}: downloaded only {written} bytes; expected at least {min_bytes}')        if expected and written != expected:            raise RuntimeError(f'{name}: Content-Length mismatch ({written} != {expected})')        part.replace(target)    except Exception:        if part.exists(): part.unlink()        raise    print('Saved:', name, f'{target.stat().st_size/1e9:.2f} GB')for folder, name, _, min_bytes in FILES:    p = MODEL_DIRS[folder]/name    assert p.exists() and p.stat().st_size >= min_bytes, f'Missing/incomplete model: {p}'print('All required model files are present.')

## 3. Запустить ComfyUI локально в Kaggle
Сервер доступен только внутри notebook runtime на 127.0.0.1. Внешний туннель намеренно не создаётся: не публикуем незащищённый ComfyUI endpoint.

In [ ]:
import socket
LOG = WORK/'comfyui.log'
log_file = open(LOG,'w')
server = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--lowvram'], cwd=COMFY, stdout=log_file, stderr=subprocess.STDOUT)
ready = False
for _ in range(180):
    if server.poll() is not None:
        print(LOG.read_text(errors='ignore')[-12000:])
        raise RuntimeError('ComfyUI завершился до запуска. См. лог выше.')
    try:
        requests.get('http://127.0.0.1:8188/system_stats', timeout=2).raise_for_status()
        ready = True; break
    except Exception: time.sleep(2)
if not ready:
    print(LOG.read_text(errors='ignore')[-12000:])
    raise TimeoutError('ComfyUI не запустился за 6 минут.')
print('ComfyUI ready:', 'http://127.0.0.1:8188')
info = requests.get('http://127.0.0.1:8188/object_info', timeout=30).json()
for node in ['UNETLoader','CLIPLoader','VAELoader','WanVaceToVideo','SaveVideo']:
    if node not in info: raise RuntimeError(f'В установленном ComfyUI отсутствует node {node}')
print('Required ComfyUI nodes are available.')

## 4. Подготовить тестовую картинку и API-workflow
Создаётся простой тестовый референс, чтобы можно было проверить весь путь без загрузки пользовательского файла.

In [ ]:
from PIL import Image, ImageDraw
ref = Image.new('RGB',(832,480),(135,190,230)); d=ImageDraw.Draw(ref)
d.rectangle((0,330,832,480), fill=(75,150,90)); d.ellipse((620,45,740,165), fill=(255,220,90))
d.rounded_rectangle((300,210,520,360), radius=28, fill=(230,100,85), outline=(40,40,50), width=5)
d.ellipse((350,245,390,285), fill='white'); d.ellipse((430,245,470,285), fill='white')
d.ellipse((365,258,378,278), fill=(20,20,30)); d.ellipse((445,258,458,278), fill=(20,20,30))
d.arc((370,275,455,330), 0, 180, fill=(30,30,30), width=5)
ref_path = COMFY/'input'/'openreels_test_reference.png'; ref_path.parent.mkdir(parents=True,exist_ok=True); ref.save(ref_path)
print('Reference image:', ref_path)

# Upload test image to ComfyUI input folder via its API.
with open(ref_path,'rb') as f:
    up = requests.post('http://127.0.0.1:8188/upload/image', files={'image':('openreels_test_reference.png',f,'image/png')}, data={'overwrite':'true'}, timeout=60)
up.raise_for_status(); uploaded = up.json(); print('Uploaded:', uploaded)

## 5. Запустить короткий VACE render
Workflow собран в API-формате из стандартных ComfyUI nodes. Если текущая версия ComfyUI поменяет схему входов, API вернёт понятную ошибку, а лог останется в `/kaggle/working/comfyui.log`.

In [ ]:
prompt = 'A charming 2D cartoon character waves hello in a sunny green meadow, gentle camera movement, clean outlines, bright colors, consistent character design, smooth animation'
negative = 'flicker, blurry, distorted face, extra limbs, text, watermark, low quality'
image_name = uploaded.get('name')
if not image_name: raise RuntimeError(f'Image upload failed: {uploaded}')

api_workflow = {
 '1': {'class_type':'UNETLoader','inputs':{'unet_name':'wan2.1_vace_1.3B_fp16.safetensors','weight_dtype':'default'}},
 '2': {'class_type':'CLIPLoader','inputs':{'clip_name':'umt5_xxl_fp8_e4m3fn_scaled.safetensors','type':'wan','device':'default'}},
 '3': {'class_type':'VAELoader','inputs':{'vae_name':'wan_2.1_vae.safetensors'}},
 '4': {'class_type':'ModelSamplingSD3','inputs':{'model':['1',0],'shift':8.0}},
 '5': {'class_type':'CLIPTextEncode','inputs':{'text':prompt,'clip':['2',0]}},
 '6': {'class_type':'CLIPTextEncode','inputs':{'text':negative,'clip':['2',0]}},
 '7': {'class_type':'LoadImage','inputs':{'image':image_name}},
 '8': {'class_type':'WanVaceToVideo','inputs':{'positive':['5',0],'negative':['6',0],'vae':['3',0],'reference_image':['7',0],'width':832,'height':480,'length':17,'batch_size':1,'strength':1.0}},
 '9': {'class_type':'KSampler','inputs':{'model':['4',0],'positive':['8',0],'negative':['8',1],'latent_image':['8',2],'seed':20261110,'steps':4,'cfg':1.0,'sampler_name':'uni_pc','scheduler':'simple','denoise':1.0}},
 '10': {'class_type':'VAEDecode','inputs':{'samples':['9',0],'vae':['3',0]}},
 '11': {'class_type':'CreateVideo','inputs':{'images':['10',0],'fps':16,'bit_depth':'auto','codec':'none','color_space':'sRGB'}},
 '12': {'class_type':'SaveVideo','inputs':{'video':['11',0],'filename_prefix':'openreels_vace_smoke','format':'auto','format.codec':'h264'}}
}

check = requests.post('http://127.0.0.1:8188/prompt',json={'prompt':api_workflow},timeout=60)
if not check.ok:
    print(json.dumps(check.json(),indent=2)[:12000])
    raise RuntimeError('ComfyUI rejected API workflow. Check object_info schemas and server log.')
prompt_id = check.json()['prompt_id']; print('Queued prompt:',prompt_id)
deadline = time.time()+60*45
history = None
while time.time()<deadline:
    time.sleep(5)
    h=requests.get(f'http://127.0.0.1:8188/history/{prompt_id}',timeout=30).json()
    history=h.get(prompt_id)
    if history:
        st=history.get('status',{})
        if st.get('status_str')=='error' or (st.get('completed') is False and st.get('messages')):
            print(json.dumps(st,indent=2)[:10000]); raise RuntimeError('VACE generation failed; see ComfyUI log.')
        if st.get('completed') is True: break
if not history or not history.get('status',{}).get('completed'):
    raise TimeoutError('Render did not complete within 45 minutes. Check GPU memory and ComfyUI log.')
print('ComfyUI completed the workflow.')

## 6. Найти MP4 и сохранить результат
Успех засчитывается только если ComfyUI действительно вернул существующий видеофайл.

In [ ]:
outputs = history.get('outputs',{})
video_items=[]
for node in outputs.values():
    for key in ('videos','gifs','files'):
        vals=node.get(key,[])
        if isinstance(vals,list):
            video_items += [v for v in vals if isinstance(v,dict) and str(v.get('filename','')).lower().endswith(('.mp4','.webm','.mov'))]
if not video_items:
    print(json.dumps(outputs,indent=2)[:12000])
    raise RuntimeError('Workflow completed but no video output was found. Check SaveVideo node and its output schema.')

v=video_items[0]
params={'filename':v['filename'],'subfolder':v.get('subfolder',''),'type':v.get('type','output')}
resp=requests.get('http://127.0.0.1:8188/view',params=params,timeout=300); resp.raise_for_status()
out=WORK/'openreels_vace_smoke.mp4'; out.write_bytes(resp.content)
if out.stat().st_size < 100_000: raise RuntimeError(f'Video file unexpectedly small: {out.stat().st_size} bytes')
print('SUCCESS — MP4 created:',out,'size:',round(out.stat().st_size/1024/1024,2),'MiB')
from IPython.display import Video, display
display(Video(str(out),embed=True))

## Результат и следующий шаг
- Если появился встроенный MP4, smoke-test прошёл на текущей GPU.
- Если ошибка CUDA OOM — попробуй другую GPU-сессию; этот ноутбук не обещает запуск на каждой T4/P100.
- Если ошибка `SaveVideo`/workflow validation — обновилась схема ComfyUI node; смотри вывод `/object_info` и `comfyui.log`.
- После завершения скачай `openreels_vace_smoke.mp4` из панели Output/Files Kaggle.
- Этот notebook пока не держит постоянный сервер и не соединяет Kaggle с OpenReels. Для этого нужен защищённый постоянный endpoint; Kaggle runtime временный.
